## 1. Logistic Regression with PyTorch

In [154]:
import torch
from torch import nn
import torch.optim as optim
from sklearn.metrics import accuracy_score

## 2. Data Processing

In [155]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

In [156]:
# Reading data

df = pd.read_csv('../data/Module_1_Lecture_2_Class_Spaceship_Titanic.csv')
df = df.set_index('PassengerId')

TARGET = 'Transported'
FEATURES = [col for col in df.columns if col != TARGET]

In [157]:
imputer_cols = ["Age", "FoodCourt", "ShoppingMall", "Spa", "VRDeck" ,"RoomService"]
imputer = SimpleImputer(strategy='median')

imputer.fit(df[imputer_cols])
df[imputer_cols] = imputer.transform(df[imputer_cols])

df["HomePlanet"] = df["HomePlanet"].fillna('Gallifrey')
df["Destination"] = df["Destination"].fillna('Skaro')

# OPTIONAL — added by me for checking
print(df["HomePlanet"].isna().sum())
print(df["Destination"].isna().sum())

df['CryoSleep_is_missing'] = df['CryoSleep'].isna().astype(int)
df['VIP_is_missing'] = df['VIP'].isna().astype(int)

df["CryoSleep"] = df["CryoSleep"].fillna(False).astype(int)
df["VIP"] = df["VIP"].fillna(False).astype(int)

# one-hot encoding
dummies = pd.get_dummies(df.loc[:, ['HomePlanet', 'Destination']], dtype=int)
# simple option: dummies = pd.get_dummies( df[["HomePlanet", "Destination"]],dtype=int)

df = pd.concat([df, dummies], axis=1)
df.drop(columns=['HomePlanet', 'Destination'], inplace=True)

df[TARGET] = df[TARGET].astype(int)

df.drop(["Name" ,"Cabin"] , axis=1 ,inplace = True)

0
0


In [158]:
print(df["CryoSleep"].isna().sum())
print(df["VIP"].isna().sum())

print(df["CryoSleep"].dtype)
print(df["VIP"].dtype)

0
0
int64
int64


In [159]:
X = df.drop(TARGET , axis =1 ).values
y = df[TARGET].values

In [160]:
X_train , X_test , y_train , y_test = train_test_split(
    X,
    y,
    random_state = 42,
    test_size =0.33,
    stratify=y
)

In [161]:
# Convert numpy arrays to PyTorch tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [162]:
# OPTIONAL — added by me for checking

print(type(X_train))
print(X_train.shape)
print(y_train.shape)
print(X_train.dtype)
print(y_train.dtype)

<class 'torch.Tensor'>
torch.Size([5824, 18])
torch.Size([5824])
torch.float32
torch.float32


## 3. Linear layer

In [163]:
m = nn.Linear(5, 3)
input = torch.randn(4, 5)
output = m(input)

print(m.weight.shape)
print(m.bias.shape)

print('Input:', input, f'shape {input.shape}', sep='\n')
print('\nOutput:', output, f'shape {output.shape}', sep='\n')

torch.Size([3, 5])
torch.Size([3])
Input:
tensor([[-0.1836, -0.8976, -0.4862, -1.3305,  0.0329],
        [ 0.6133,  1.4898, -0.1787, -1.9354,  0.1788],
        [ 0.6224, -0.4030,  0.0072, -0.7280,  0.5507],
        [ 0.0115, -0.9384,  0.2450, -0.1389,  0.0315]])
shape torch.Size([4, 5])

Output:
tensor([[-0.2227,  0.5109, -0.0033],
        [ 0.0249, -0.1926, -0.5629],
        [ 0.3479,  0.3350,  0.1029],
        [ 0.0238,  0.4400,  0.2719]], grad_fn=<AddmmBackward0>)
shape torch.Size([4, 3])


## 4. Sigmoid activation

In [164]:
t = torch.randn(4)
print('Input: ', t)
print('Applying sigmoid: ', torch.sigmoid(t))

Input:  tensor([ 0.6294, -0.2725,  1.2503, -0.7962])
Applying sigmoid:  tensor([0.6523, 0.4323, 0.7774, 0.3108])


## 5. Logistic Regression model

In [165]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.linear(x)
        out = self.sigmoid(out)
        return out

## 6. Create a model

In [166]:
input_dim = X_train.shape[1]
model = LogisticRegression(input_dim)

In [167]:
print(model)

LogisticRegression(
  (linear): Linear(in_features=18, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


## 7. Loss function and optimizer

In [168]:
criterion = nn.BCELoss()

In [169]:
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [170]:
model.parameters

<bound method Module.parameters of LogisticRegression(
  (linear): Linear(in_features=18, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)>

In [171]:
# OPTIONAL — added by me for understanding

for name, param in model.named_parameters():
    print(name, param.shape)

linear.weight torch.Size([1, 18])
linear.bias torch.Size([1])


## 8. Train the model

In [172]:
# Train the model

num_epochs = 50

for epoch in range(num_epochs):
    # Forward pass
    outputs = model(X_train)
    loss = criterion(outputs.squeeze(), y_train)
    
    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [5/50], Loss: 29.0973
Epoch [10/50], Loss: 27.6381
Epoch [15/50], Loss: 12.2008
Epoch [20/50], Loss: 12.3535
Epoch [25/50], Loss: 12.2040
Epoch [30/50], Loss: 12.3849
Epoch [35/50], Loss: 12.1571
Epoch [40/50], Loss: 12.7995
Epoch [45/50], Loss: 12.8115
Epoch [50/50], Loss: 12.6586


In [173]:
# OPTIONAL — added by me for understanding

print(model.linear.weight.grad)
print(model.linear.bias.grad)

tensor([[-2.5556e-02,  3.6853e-03, -6.2113e-04, -6.3071e-02,  8.9952e-01,
          7.1120e-02, -2.9185e-02,  5.7398e-01,  4.1651e-04,  3.4087e-04,
          8.0297e-03, -2.0471e-02, -8.1158e-04, -1.1440e-02, -1.6580e-02,
         -1.5749e-04, -8.0334e-04, -7.1512e-03]])
tensor([-0.0247])


## 9. Evaluate the model

In [174]:
# Test the model

with torch.no_grad():
    y_pred = model(X_test).squeeze().numpy().round()

accuracy_score(y_test, y_pred)

0.7870338096897874